In [ ]:
from google.colab import files

uploaded = files.upload()

In [ ]:
import zipfile
import os

with zipfile.ZipFile('Student folder.zip', 'r') as zip_ref:
    zip_ref.extractall('/content/Student folder')

In [ ]:
!pip -q install folium networkx pyproj pyarrow scikit-learn scipy joblib fastapi uvicorn

import os, glob, json, random, hashlib, time, logging
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import networkx as nx, folium, joblib
from pyproj import Transformer
from scipy.spatial import cKDTree
from sklearn.preprocessing import StandardScaler

In [ ]:
ROOT = '/content/Student folder'

TRAIN = f'{ROOT}/train'
TEST = f'{ROOT}/student_test'

DATA, EXP, ART = f'{ROOT}/data', f'{ROOT}/experiments', f'{ROOT}/models/artifacts'
FIG, MAPS = f'{ROOT}/visualisations/figures', f'{ROOT}/visualisations/maps'

for d in [DATA, EXP, ART, FIG, MAPS, f'{ROOT}/api', f'{ROOT}/logs']:
    os.makedirs(d, exist_ok=True)
SEED = 42
EXT  = ['track', 'nodes', 'arcs', 'route']

In [ ]:

# Raw dataset paths
ROOT = '/content/Student folder/Student folder'

TRAIN = f'{ROOT}/train'
TEST = f'{ROOT}/student_test'

# Output paths (separate from raw dataset)
OUT = '/content/PathSnap'

DATA = f'{OUT}/data'
EXP = f'{OUT}/experiments'
ART = f'{OUT}/models/artifacts'
FIG = f'{OUT}/visualisations/figures'
MAPS = f'{OUT}/visualisations/maps'

for d in [DATA, EXP, ART, FIG, MAPS,
          f'{OUT}/api', f'{OUT}/logs']:
    os.makedirs(d, exist_ok=True)

SEED = 42
EXT = ['track', 'nodes', 'arcs', 'route']

print("Train folder:", os.path.isdir(TRAIN))
print("Student test folder:", os.path.isdir(TEST))
print("Output folder:", OUT)


In [ ]:

import os

print("Contents of /content:")
for item in os.listdir('/content'):
    print(item)

print("\nContents of Student folder:")
for item in os.listdir('/content/Student folder'):
    print(item)


In [ ]:

RAW = '/content/Student folder/Student folder'

print("Contents of nested Student folder:")
for item in os.listdir(RAW):
    print(item)


In [ ]:

for split_name, split_path in [('train', TRAIN), ('student_test', TEST)]:
    print(f'\n--- {split_name.upper()} ---')

    items = sorted(os.listdir(split_path))
    print("Number of top-level items:", len(items))

    for item in items[:8]:
        path = os.path.join(split_path, item)
        if os.path.isdir(path):
            print(f"{item}/ -> {os.listdir(path)[:8]}")
        else:
            print(item)


In [ ]:

import pandas as pd
from pathlib import Path

records = []

for split_name, split_path in [('train', TRAIN), ('student_test', TEST)]:
    for trip_dir in sorted(Path(split_path).iterdir()):
        if not trip_dir.is_dir():
            continue

        record = {
            'split': split_name,
            'trip_id': trip_dir.name
        }

        for ext in EXT:
            files = list(trip_dir.glob(f'*.{ext}'))
            record[f'has_{ext}'] = len(files) > 0
            record[f'{ext}_count'] = len(files)
            record[f'{ext}_size_bytes'] = (
                files[0].stat().st_size if files else 0
            )

        records.append(record)

inventory = pd.DataFrame(records)

inventory.to_csv(f'{EXP}/inventory.csv', index=False)

print("Total trips:", len(inventory))
print("\nTrips by partition:")
print(inventory.groupby('split').size())

print("\nFile availability:")
print(inventory.groupby('split')[
    [f'has_{ext}' for ext in EXT]
].sum())

print("\nMissing files:")
print(inventory[
    ~inventory[[f'has_{ext}' for ext in EXT]].all(axis=1)
][['split', 'trip_id']])

print("\nSaved:", f'{EXP}/inventory.csv')


In [ ]:

import numpy as np

quality_records = []

for split_name, split_path in [('train', TRAIN), ('student_test', TEST)]:
    for trip_dir in sorted(Path(split_path).iterdir()):
        if not trip_dir.is_dir():
            continue

        track_file = next(trip_dir.glob('*.track'))

        df = pd.read_csv(
            track_file,
            sep=r'\s+',
            header=None,
            names=['lon', 'lat', 'timestamp']
        )

        nan_rows = df.isna().any(axis=1).sum()

        valid = df.dropna().copy()

        time_diff = valid['timestamp'].diff().dropna()

        quality_records.append({
            'split': split_name,
            'trip_id': trip_dir.name,
            'total_rows': len(df),
            'nan_rows': int(nan_rows),
            'unsorted_timestamp_steps': int((time_diff < 0).sum()),
            'duplicate_timestamp_steps': int((time_diff == 0).sum()),
            'one_second_step_share': (
                float(np.isclose(time_diff, 1).mean())
                if len(time_diff) else np.nan
            ),
            'max_gap_seconds': (
                float(time_diff.max()) if len(time_diff) else np.nan
            ),
            'out_of_range_coordinates': int(
                (
                    ~valid['lon'].between(-180, 180) |
                    ~valid['lat'].between(-90, 90)
                ).sum()
            )
        })

quality = pd.DataFrame(quality_records)

quality.to_csv(f'{EXP}/data_quality.csv', index=False)

print("Total trips analysed:", len(quality))
print("\nQuality summary:")
print(quality.groupby('split').agg({
    'total_rows': 'sum',
    'nan_rows': 'sum',
    'unsorted_timestamp_steps': 'sum',
    'duplicate_timestamp_steps': 'sum',
    'one_second_step_share': 'mean',
    'max_gap_seconds': 'max',
    'out_of_range_coordinates': 'sum'
}).round(4))

print("\nSaved:", f'{EXP}/data_quality.csv')


In [ ]:

swap_records = []

for split_name, split_path in [('train', TRAIN), ('student_test', TEST)]:
    for trip_dir in sorted(Path(split_path).iterdir()):
        if not trip_dir.is_dir():
            continue

        track_file = next(trip_dir.glob('*.track'))
        nodes_file = next(trip_dir.glob('*.nodes'))

        track = pd.read_csv(
            track_file, sep=r'\s+', header=None,
            names=['lon', 'lat', 'timestamp']
        )

        nodes = pd.read_csv(
            nodes_file, sep=r'\s+', header=None,
            names=['lon', 'lat']
        )

        lon_min, lon_max = nodes['lon'].min(), nodes['lon'].max()
        lat_min, lat_max = nodes['lat'].min(), nodes['lat'].max()

        valid_track = track.dropna(subset=['lon', 'lat'])

        inside = (
            valid_track['lon'].between(lon_min, lon_max) &
            valid_track['lat'].between(lat_min, lat_max)
        )

        swap_records.append({
            'split': split_name,
            'trip_id': trip_dir.name,
            'track_points': len(valid_track),
            'points_inside_bbox': int(inside.sum()),
            'points_outside_bbox': int((~inside).sum()),
            'inside_share': float(inside.mean()),
            'track_lon_min': valid_track['lon'].min(),
            'track_lon_max': valid_track['lon'].max(),
            'track_lat_min': valid_track['lat'].min(),
            'track_lat_max': valid_track['lat'].max(),
            'network_lon_min': lon_min,
            'network_lon_max': lon_max,
            'network_lat_min': lat_min,
            'network_lat_max': lat_max
        })

swap_check = pd.DataFrame(swap_records)

swap_check.to_csv(f'{EXP}/swap_check.csv', index=False)

print("Trips checked:", len(swap_check))
print("\nBounding-box summary:")
print(swap_check.groupby('split').agg({
    'track_points': 'sum',
    'points_inside_bbox': 'sum',
    'points_outside_bbox': 'sum',
    'inside_share': 'mean'
}).round(4))

print("\nTrips with outside points:")
print(swap_check[swap_check['points_outside_bbox'] > 0][
    ['split', 'trip_id', 'points_outside_bbox']
])

print("\nSaved:", f'{EXP}/swap_check.csv')


In [ ]:

import hashlib
import networkx as nx

network_records = []

def file_hash(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(65536), b''):
            h.update(chunk)
    return h.hexdigest()

for split_name, split_path in [('train', TRAIN), ('student_test', TEST)]:
    for trip_dir in sorted(Path(split_path).iterdir()):
        if not trip_dir.is_dir():
            continue

        nodes_file = next(trip_dir.glob('*.nodes'))
        arcs_file = next(trip_dir.glob('*.arcs'))

        nodes = pd.read_csv(
            nodes_file, sep=r'\s+', header=None,
            names=['lon', 'lat']
        )

        arcs = pd.read_csv(
            arcs_file, sep=r'\s+', header=None,
            names=['from_node', 'to_node']
        )

        n_nodes = len(nodes)

        valid_mask = (
            arcs['from_node'].between(0, n_nodes - 1) &
            arcs['to_node'].between(0, n_nodes - 1)
        )

        valid_arcs = arcs[valid_mask]

        self_loops = (
            valid_arcs['from_node'] == valid_arcs['to_node']
        ).sum()

        duplicate_arcs = valid_arcs.duplicated().sum()

        G = nx.Graph()
        G.add_nodes_from(range(n_nodes))
        G.add_edges_from(
            valid_arcs[['from_node', 'to_node']].itertuples(
                index=False, name=None
            )
        )

        network_records.append({
            'split': split_name,
            'trip_id': trip_dir.name,
            'node_count': n_nodes,
            'arc_count': len(arcs),
            'invalid_arcs': int((~valid_mask).sum()),
            'self_loops': int(self_loops),
            'duplicate_arcs': int(duplicate_arcs),
            'isolated_nodes': len(list(nx.isolates(G))),
            'connected_components': nx.number_connected_components(G),
            'nodes_hash': file_hash(nodes_file),
            'arcs_hash': file_hash(arcs_file)
        })

network_summary = pd.DataFrame(network_records)

network_summary.to_csv(
    f'{EXP}/network_summary.csv', index=False
)

print("Networks analysed:", len(network_summary))

print("\nNetwork summary:")
print(network_summary.groupby('split').agg({
    'node_count': ['min', 'max', 'mean'],
    'arc_count': ['min', 'max', 'mean'],
    'invalid_arcs': 'sum',
    'self_loops': 'sum',
    'duplicate_arcs': 'sum',
    'isolated_nodes': 'sum',
    'connected_components': ['min', 'max']
}).round(2))

print("\nUnique network hash combinations:")
print(network_summary.groupby(
    ['nodes_hash', 'arcs_hash']
).size().to_string())

print("\nSaved:", f'{EXP}/network_summary.csv')


In [ ]:
from pyproj import CRS
import json

# Read coordinates from training tracks only
train_lons, train_lats = [], []

for trip_dir in sorted(Path(TRAIN).iterdir()):
    if not trip_dir.is_dir():
        continue

    track_file = next(trip_dir.glob('*.track'))
    track = pd.read_csv(
        track_file, sep=r'\s+', header=None,
        names=['lon', 'lat', 'timestamp']
    )

    train_lons.extend(track['lon'].dropna().tolist())
    train_lats.extend(track['lat'].dropna().tolist())

min_lon, max_lon = min(train_lons), max(train_lons)
min_lat, max_lat = min(train_lats), max(train_lats)

center_lon = (min_lon + max_lon) / 2
center_lat = (min_lat + max_lat) / 2

utm_zone = int((center_lon + 180) // 6) + 1
epsg = (32600 if center_lat >= 0 else 32700) + utm_zone

projection = {
    'source_crs': 'EPSG:4326',
    'target_crs': f'EPSG:{epsg}',
    'utm_zone': utm_zone,
    'bounds_train_only': {
        'min_lon': min_lon,
        'max_lon': max_lon,
        'min_lat': min_lat,
        'max_lat': max_lat
    },
    'center_lon': center_lon,
    'center_lat': center_lat
}

with open(f'{EXP}/projection.json', 'w') as f:
    json.dump(projection, f, indent=4)

print(json.dumps(projection, indent=4))
print(f"\nSaved: {EXP}/projection.json")

In [ ]:
records = []

for split_name, split_path in [('train', TRAIN), ('student_test', TEST)]:
    for trip_dir in sorted(Path(split_path).iterdir()):
        if not trip_dir.is_dir():
            continue

        track_file = next(trip_dir.glob('*.track'))
        track = pd.read_csv(
            track_file, sep=r'\s+', header=None,
            names=['lon', 'lat', 'timestamp']
        )

        records.append({
            'split': split_name,
            'trip_id': trip_dir.name,
            'points': len(track),
            'min_lon': track['lon'].min(),
            'max_lon': track['lon'].max(),
            'min_lat': track['lat'].min(),
            'max_lat': track['lat'].max(),
            'center_lon': track['lon'].mean(),
            'center_lat': track['lat'].mean()
        })

trip_extent = pd.DataFrame(records)
trip_extent.to_csv(
    f'{EXP}/trip_geographic_extent.csv', index=False
)

print("Training trip geographic centers:")
print(
    trip_extent[trip_extent['split'] == 'train'][
        ['trip_id', 'center_lon', 'center_lat']
    ].to_string(index=False)
)

print("\nOverall center ranges:")
print(
    trip_extent.groupby('split')[
        ['center_lon', 'center_lat']
    ].agg(['min', 'max']).round(4)
)

print(f"\nSaved: {EXP}/trip_geographic_extent.csv")

In [ ]:
print("Trips with largest geographic spans:")

trip_extent['lon_span'] = (
    trip_extent['max_lon'] - trip_extent['min_lon']
)
trip_extent['lat_span'] = (
    trip_extent['max_lat'] - trip_extent['min_lat']
)

print(
    trip_extent.sort_values(
        ['lon_span', 'lat_span'], ascending=False
    )[
        ['split', 'trip_id', 'points',
         'lon_span', 'lat_span',
         'center_lon', 'center_lat']
    ].head(15).to_string(index=False)
)

print("\nTrips spanning more than 2 degrees:")
print(
    trip_extent[
        (trip_extent['lon_span'] > 2) |
        (trip_extent['lat_span'] > 2)
    ][['split', 'trip_id', 'lon_span', 'lat_span']]
    .to_string(index=False)
)

In [ ]:
def get_utm_epsg(lon, lat):
    zone = int((lon + 180) // 6) + 1
    zone = max(1, min(zone, 60))
    return (32600 if lat >= 0 else 32700) + zone

projection_records = []

for _, row in trip_extent.iterrows():
    epsg = get_utm_epsg(row['center_lon'], row['center_lat'])

    projection_records.append({
        'split': row['split'],
        'trip_id': row['trip_id'],
        'center_lon': row['center_lon'],
        'center_lat': row['center_lat'],
        'utm_zone': (epsg % 100),
        'hemisphere': 'north' if row['center_lat'] >= 0 else 'south',
        'epsg': epsg,
        'crs': f'EPSG:{epsg}'
    })

projections_by_trip = pd.DataFrame(projection_records)

projections_by_trip.to_csv(
    f'{EXP}/projections_by_trip.csv', index=False
)

print("Projection counts by partition:")
print(
    projections_by_trip.groupby(['split', 'crs'])
    .size().to_string()
)

print("\nSample projection assignments:")
print(projections_by_trip.head(10).to_string(index=False))

print(f"\nSaved: {EXP}/projections_by_trip.csv")

In [ ]:
from pyproj import Transformer
import numpy as np

rng = np.random.default_rng(SEED)
distance_checks = []

for _, row in projections_by_trip[
    projections_by_trip['split'] == 'train'
].iterrows():

    trip_dir = Path(TRAIN) / row['trip_id']
    track_file = next(trip_dir.glob('*.track'))

    track = pd.read_csv(
        track_file, sep=r'\s+', header=None,
        names=['lon', 'lat', 'timestamp']
    ).dropna(subset=['lon', 'lat'])

    lon = track['lon'].to_numpy()
    lat = track['lat'].to_numpy()

    if len(lon) < 2:
        continue

    transformer = Transformer.from_crs(
        'EPSG:4326', row['crs'], always_xy=True
    )

    x, y = transformer.transform(lon, lat)

    projected = np.hypot(np.diff(x), np.diff(y))

    lon1, lat1 = np.radians(lon[:-1]), np.radians(lat[:-1])
    lon2, lat2 = np.radians(lon[1:]), np.radians(lat[1:])

    dlon = lon2 - lon1
    dlat = lat2 - lat1

    a = (
        np.sin(dlat / 2) ** 2
        + np.cos(lat1) * np.cos(lat2)
        * np.sin(dlon / 2) ** 2
    )

    haversine = (
        2 * 6371008.8 * np.arcsin(np.sqrt(np.clip(a, 0, 1)))
    )

    valid = np.isfinite(projected) & np.isfinite(haversine) & (haversine > 0)

    if not valid.any():
        continue

    abs_error = np.abs(projected[valid] - haversine[valid])
    relative_error = abs_error / haversine[valid] * 100

    distance_checks.append({
        'trip_id': row['trip_id'],
        'epsg': row['epsg'],
        'pairs_checked': int(valid.sum()),
        'mean_absolute_error_m': abs_error.mean(),
        'max_absolute_error_m': abs_error.max(),
        'mean_relative_error_pct': relative_error.mean(),
        'max_relative_error_pct': relative_error.max()
    })

distance_check_df = pd.DataFrame(distance_checks)

distance_check_df.to_csv(
    f'{EXP}/projection_distance_check.csv', index=False
)

print("Training projection validation:")
print(distance_check_df.describe().round(6).to_string())

print("\nWorst trips by mean relative error:")
print(
    distance_check_df.nlargest(
        5, 'mean_relative_error_pct'
    ).to_string(index=False)
)

print(f"\nSaved: {EXP}/projection_distance_check.csv")

In [ ]:
projection_config = {
    'source_crs': 'EPSG:4326',
    'projection_method': 'per_trip_utm',
    'zone_selection': 'calculated from each trip geographic centre',
    'hemisphere_selection': 'based on trip centre latitude',
    'training_trips_used_for_validation': len(distance_check_df),
    'validation': {
        'method': 'UTM Euclidean distance vs haversine distance',
        'mean_absolute_error_m': float(
            distance_check_df['mean_absolute_error_m'].mean()
        ),
        'max_absolute_error_m': float(
            distance_check_df['max_absolute_error_m'].max()
        ),
        'mean_relative_error_pct': float(
            distance_check_df['mean_relative_error_pct'].mean()
        ),
        'max_mean_relative_error_pct': float(
            distance_check_df['mean_relative_error_pct'].max()
        )
    },
    'details_file': 'projections_by_trip.csv',
    'distance_validation_file': 'projection_distance_check.csv'
}

with open(f'{EXP}/projection.json', 'w') as f:
    json.dump(projection_config, f, indent=4)

print(json.dumps(projection_config, indent=4))
print(f"\nUpdated: {EXP}/projection.json")

In [ ]:
network_checks = []

for split_name, split_path in [('train', TRAIN), ('student_test', TEST)]:
    for trip_dir in sorted(Path(split_path).iterdir()):
        if not trip_dir.is_dir():
            continue

        track_file = next(trip_dir.glob('*.track'))
        nodes_file = next(trip_dir.glob('*.nodes'))

        track = pd.read_csv(
            track_file, sep=r'\s+', header=None,
            names=['lon', 'lat', 'timestamp']
        )

        nodes = pd.read_csv(
            nodes_file, sep=r'\s+', header=None,
            names=['lon', 'lat']
        )

        network_checks.append({
            'split': split_name,
            'trip_id': trip_dir.name,
            'track_points': len(track),
            'network_nodes': len(nodes),
            'track_min_lon': track['lon'].min(),
            'track_max_lon': track['lon'].max(),
            'track_min_lat': track['lat'].min(),
            'track_max_lat': track['lat'].max(),
            'nodes_min_lon': nodes['lon'].min(),
            'nodes_max_lon': nodes['lon'].max(),
            'nodes_min_lat': nodes['lat'].min(),
            'nodes_max_lat': nodes['lat'].max()
        })

network_check_df = pd.DataFrame(network_checks)

network_check_df['bbox_overlap'] = (
    (network_check_df['track_max_lon'] >= network_check_df['nodes_min_lon']) &
    (network_check_df['track_min_lon'] <= network_check_df['nodes_max_lon']) &
    (network_check_df['track_max_lat'] >= network_check_df['nodes_min_lat']) &
    (network_check_df['track_min_lat'] <= network_check_df['nodes_max_lat'])
)

network_check_df.to_csv(
    f'{EXP}/network_coordinate_check.csv', index=False
)

print("Bounding-box compatibility:")
print(
    network_check_df.groupby('split')['bbox_overlap']
    .agg(['count', 'sum'])
)

print("\nTrips without bounding-box overlap:")
print(
    network_check_df[~network_check_df['bbox_overlap']]
    [['split', 'trip_id']]
    .to_string(index=False)
)

print(f"\nSaved: {EXP}/network_coordinate_check.csv")

In [ ]:
from scipy.spatial import cKDTree
from pyproj import Transformer

# Select one training trip for validation
trip_id = sorted([
    d.name for d in Path(TRAIN).iterdir() if d.is_dir()
])[0]

trip_dir = Path(TRAIN) / trip_id

nodes = pd.read_csv(
    next(trip_dir.glob('*.nodes')),
    sep=r'\s+', header=None,
    names=['lon', 'lat']
)

arcs = pd.read_csv(
    next(trip_dir.glob('*.arcs')),
    sep=r'\s+', header=None,
    names=['from_node', 'to_node']
)

# Use this trip's assigned projection
proj_row = projections_by_trip[
    (projections_by_trip['split'] == 'train') &
    (projections_by_trip['trip_id'] == trip_id)
].iloc[0]

transformer = Transformer.from_crs(
    'EPSG:4326', proj_row['crs'], always_xy=True
)

node_x, node_y = transformer.transform(
    nodes['lon'].to_numpy(),
    nodes['lat'].to_numpy()
)

# Remove invalid arc references before indexing
valid = (
    arcs['from_node'].between(0, len(nodes)-1) &
    arcs['to_node'].between(0, len(nodes)-1)
)

valid_arcs = arcs.loc[valid].copy()
valid_arcs['arc_id'] = valid_arcs.index

from_ids = valid_arcs['from_node'].to_numpy(dtype=int)
to_ids = valid_arcs['to_node'].to_numpy(dtype=int)

# Arc endpoint coordinates
x1, y1 = node_x[from_ids], node_y[from_ids]
x2, y2 = node_x[to_ids], node_y[to_ids]

# Midpoints and lengths
mid_x = (x1 + x2) / 2
mid_y = (y1 + y2) / 2
arc_lengths = np.hypot(x2 - x1, y2 - y1)

# Build KD-tree
midpoint_tree = cKDTree(np.column_stack([mid_x, mid_y]))

print("Trip:", trip_id)
print("Nodes:", len(nodes))
print("Original arcs:", len(arcs))
print("Valid arcs:", len(valid_arcs))
print("Invalid arcs:", int((~valid).sum()))
print("KD-tree entries:", len(midpoint_tree.data))
print("Mean arc length (m):", round(arc_lengths.mean(), 3))
print("Maximum arc length (m):", round(arc_lengths.max(), 3))

In [ ]:
track = pd.read_csv(
    next(trip_dir.glob('*.track')),
    sep=r'\s+', header=None,
    names=['lon', 'lat', 'timestamp']
)

gps_x, gps_y = transformer.transform(
    track['lon'].to_numpy(),
    track['lat'].to_numpy()
)

gps_xy = np.column_stack([gps_x, gps_y])

pilot_radius = 50

candidate_indices = midpoint_tree.query_ball_point(
    gps_xy, r=pilot_radius
)

candidate_counts = np.array([
    len(indices) for indices in candidate_indices
])

print("Pilot radius:", pilot_radius, "m")
print("GPS observations:", len(track))
print("Observations with candidates:", np.sum(candidate_counts > 0))
print("Observations without candidates:", np.sum(candidate_counts == 0))
print("Mean candidates per observation:", round(candidate_counts.mean(), 2))
print("Median candidates:", np.median(candidate_counts))
print("Maximum candidates:", candidate_counts.max())
print("95th percentile:", np.percentile(candidate_counts, 95))

In [ ]:
# Sample each road segment at intervals of at most 10 metres
sample_spacing = 10.0

sample_x = []
sample_y = []
sample_arc_ids = []

for i, arc_id in enumerate(valid_arcs['arc_id'].to_numpy()):
    length = arc_lengths[i]

    n_segments = max(1, int(np.ceil(length / sample_spacing)))

    fractions = np.linspace(0, 1, n_segments + 1)

    sample_x.extend(x1[i] + fractions * (x2[i] - x1[i]))
    sample_y.extend(y1[i] + fractions * (y2[i] - y1[i]))
    sample_arc_ids.extend([arc_id] * len(fractions))

sample_points = np.column_stack([sample_x, sample_y])
sample_arc_ids = np.asarray(sample_arc_ids)

dense_tree = cKDTree(sample_points)

print("Original arcs:", len(valid_arcs))
print("Densified index points:", len(sample_points))
print("Sampling interval:", sample_spacing, "m")
print("Mean points per arc:", round(len(sample_points) / len(valid_arcs), 2))

In [ ]:
midpoint_results = midpoint_tree.query_ball_point(
    gps_xy, r=50
)

dense_results = dense_tree.query_ball_point(
    gps_xy, r=50
)

midpoint_unique = [
    len(set(valid_arcs['arc_id'].to_numpy()[indices]))
    for indices in midpoint_results
]

dense_unique = [
    len(set(sample_arc_ids[indices]))
    for indices in dense_results
]

comparison = pd.DataFrame({
    'method': ['midpoint', 'densified'],
    'observations_with_candidates': [
        sum(x > 0 for x in midpoint_unique),
        sum(x > 0 for x in dense_unique)
    ],
    'mean_unique_candidates': [
        np.mean(midpoint_unique),
        np.mean(dense_unique)
    ],
    'median_unique_candidates': [
        np.median(midpoint_unique),
        np.median(dense_unique)
    ],
    'max_unique_candidates': [
        max(midpoint_unique),
        max(dense_unique)
    ]
})

print(comparison.to_string(index=False))

In [ ]:
# Retrieve nearby sample points, then map them to unique original arcs
nearby_samples = dense_tree.query_ball_point(gps_xy, r=50)

candidate_records = []

for obs_idx, sample_indices in enumerate(nearby_samples):
    arc_ids = np.unique(sample_arc_ids[sample_indices])

    for arc_id in arc_ids:
        arc_pos = np.searchsorted(
            valid_arcs['arc_id'].to_numpy(), arc_id
        )

        dx = x2[arc_pos] - x1[arc_pos]
        dy = y2[arc_pos] - y1[arc_pos]

        denominator = dx**2 + dy**2

        if denominator == 0:
            t = 0.0
        else:
            t = (
                (gps_x[obs_idx] - x1[arc_pos]) * dx
                + (gps_y[obs_idx] - y1[arc_pos]) * dy
            ) / denominator

        t = np.clip(t, 0, 1)

        closest_x = x1[arc_pos] + t * dx
        closest_y = y1[arc_pos] + t * dy

        distance = np.hypot(
            gps_x[obs_idx] - closest_x,
            gps_y[obs_idx] - closest_y
        )

        if distance <= 50:
            candidate_records.append({
                'obs_idx': obs_idx,
                'arc_id': int(arc_id),
                'perp_distance_m': distance,
                'projection_fraction': t
            })

pilot_candidates = pd.DataFrame(candidate_records)

print("Exact candidates:", len(pilot_candidates))
print("Observations with candidates:",
      pilot_candidates['obs_idx'].nunique())
print("Mean exact candidates per observation:",
      round(len(pilot_candidates) / len(track), 2))
print("Maximum exact distance:",
      round(pilot_candidates['perp_distance_m'].max(), 3), "m")
print("Projection fraction range:",
      round(pilot_candidates['projection_fraction'].min(), 4),
      "to",
      round(pilot_candidates['projection_fraction'].max(), 4))

In [ ]:
def generate_candidates(
    gps_x, gps_y, x1, y1, x2, y2,
    arc_ids, radius, sample_spacing=10.0
):
    sample_x, sample_y, sample_arc_ids = [], [], []

    for i, arc_id in enumerate(arc_ids):
        length = np.hypot(x2[i] - x1[i], y2[i] - y1[i])
        n = max(1, int(np.ceil(length / sample_spacing)))
        fractions = np.linspace(0, 1, n + 1)

        sample_x.extend(x1[i] + fractions * (x2[i] - x1[i]))
        sample_y.extend(y1[i] + fractions * (y2[i] - y1[i]))
        sample_arc_ids.extend([arc_id] * len(fractions))

    sample_points = np.column_stack([sample_x, sample_y])
    sample_arc_ids = np.asarray(sample_arc_ids)

    tree = cKDTree(sample_points)
    gps_xy = np.column_stack([gps_x, gps_y])

    # Safety margin for the maximum distance to a sample
    search_radius = radius + sample_spacing / 2
    nearby = tree.query_ball_point(gps_xy, r=search_radius)

    arc_positions = {
        int(arc_id): i for i, arc_id in enumerate(arc_ids)
    }

    records = []

    for obs_idx, sample_indices in enumerate(nearby):
        nearby_arcs = np.unique(sample_arc_ids[sample_indices])

        for arc_id in nearby_arcs:
            i = arc_positions[int(arc_id)]
            dx, dy = x2[i] - x1[i], y2[i] - y1[i]
            denominator = dx**2 + dy**2

            t = 0.0 if denominator == 0 else (
                (gps_x[obs_idx] - x1[i]) * dx
                + (gps_y[obs_idx] - y1[i]) * dy
            ) / denominator

            t = np.clip(t, 0, 1)
            closest_x = x1[i] + t * dx
            closest_y = y1[i] + t * dy

            distance = np.hypot(
                gps_x[obs_idx] - closest_x,
                gps_y[obs_idx] - closest_y
            )

            if distance <= radius:
                records.append({
                    'obs_idx': obs_idx,
                    'arc_id': int(arc_id),
                    'perp_distance_m': float(distance),
                    'projection_fraction': float(t)
                })

    return pd.DataFrame(records, columns=[
        'obs_idx', 'arc_id',
        'perp_distance_m', 'projection_fraction'
    ])

print("Corrected candidate-generation function defined.")
print("Search margin: sample_spacing / 2")

In [ ]:
pilot_candidates_v2 = generate_candidates(
    gps_x, gps_y,
    x1, y1, x2, y2,
    valid_arcs['arc_id'].to_numpy(),
    radius=50,
    sample_spacing=10
)

print("Original pilot candidates:", len(pilot_candidates))
print("Corrected function candidates:", len(pilot_candidates_v2))

print("\nObservations with candidates:",
      pilot_candidates_v2['obs_idx'].nunique(), "/", len(track))

print("Mean candidates per observation:",
      round(len(pilot_candidates_v2) / len(track), 2))

print("Maximum perpendicular distance:",
      round(pilot_candidates_v2['perp_distance_m'].max(), 3), "m")

print("Duplicate observation-arc pairs:",
      pilot_candidates_v2.duplicated(['obs_idx', 'arc_id']).sum())

In [ ]:
radii = [10, 15, 20, 30, 50]
tuning_records = []

train_trip_dirs = sorted(
    [d for d in Path(TRAIN).iterdir() if d.is_dir()]
)

for trip_num, trip_dir in enumerate(train_trip_dirs, 1):
    trip_id = trip_dir.name

    track = pd.read_csv(
        next(trip_dir.glob('*.track')),
        sep=r'\s+', header=None,
        names=['lon', 'lat', 'timestamp']
    )

    nodes = pd.read_csv(
        next(trip_dir.glob('*.nodes')),
        sep=r'\s+', header=None,
        names=['lon', 'lat']
    )

    arcs = pd.read_csv(
        next(trip_dir.glob('*.arcs')),
        sep=r'\s+', header=None,
        names=['from_node', 'to_node']
    )

    route = pd.read_csv(
        next(trip_dir.glob('*.route')),
        sep=r'\s+', header=None
    ).iloc[:, 0].dropna().astype(int)

    proj_row = projections_by_trip[
        (projections_by_trip['split'] == 'train') &
        (projections_by_trip['trip_id'] == trip_id)
    ].iloc[0]

    transformer = Transformer.from_crs(
        'EPSG:4326', proj_row['crs'], always_xy=True
    )

    node_x, node_y = transformer.transform(
        nodes['lon'].to_numpy(), nodes['lat'].to_numpy()
    )
    gps_x, gps_y = transformer.transform(
        track['lon'].to_numpy(), track['lat'].to_numpy()
    )

    valid = (
        arcs['from_node'].between(0, len(nodes)-1) &
        arcs['to_node'].between(0, len(nodes)-1)
    )
    valid_arcs = arcs.loc[valid]
    arc_ids = valid_arcs.index.to_numpy()

    from_ids = valid_arcs['from_node'].to_numpy(dtype=int)
    to_ids = valid_arcs['to_node'].to_numpy(dtype=int)

    x1, y1 = node_x[from_ids], node_y[from_ids]
    x2, y2 = node_x[to_ids], node_y[to_ids]

    candidates = generate_candidates(
        gps_x, gps_y, x1, y1, x2, y2,
        arc_ids, radius=50, sample_spacing=10
    )

    route_ids = set(route.tolist())
    route_candidates = candidates[
        candidates['arc_id'].isin(route_ids)
    ]

    matched_obs = set(route_candidates['obs_idx'].unique())
    total_obs = len(track)

    for radius in radii:
        within_radius = route_candidates[
            route_candidates['perp_distance_m'] <= radius
        ]

        covered = within_radius['obs_idx'].nunique()
        missed = total_obs - covered

        tuning_records.append({
            'trip_id': trip_id,
            'radius_m': radius,
            'observations': total_obs,
            'covered_observations': covered,
            'missed_observations': missed,
            'miss_rate_pct': missed / total_obs * 100
        })

    if trip_num % 10 == 0:
        print(f"Processed {trip_num}/{len(train_trip_dirs)} training trips")

tuning_df = pd.DataFrame(tuning_records)

tuning_summary = tuning_df.groupby('radius_m').agg(
    trips=('trip_id', 'nunique'),
    observations=('observations', 'sum'),
    missed_observations=('missed_observations', 'sum')
).reset_index()

tuning_summary['miss_rate_pct'] = (
    tuning_summary['missed_observations'] /
    tuning_summary['observations'] * 100
)

tuning_summary.to_csv(
    f'{EXP}/R_tuning_train.csv', index=False
)

print("\nRadius tuning results:")
print(tuning_summary.to_string(index=False))

eligible = tuning_summary[
    tuning_summary['miss_rate_pct'] <= 2
]

if len(eligible):
    selected_radius = int(eligible.iloc[0]['radius_m'])
    print(f"\nSelected radius: {selected_radius} m")
else:
    selected_radius = None
    print("\nNo tested radius achieved the required 2% miss rate.")

print(f"\nSaved: {EXP}/R_tuning_train.csv")

In [ ]:
display(tuning_summary)

eligible = tuning_summary[
    tuning_summary['miss_rate_pct'] <= 2
]

if len(eligible):
    SELECTED_R = int(eligible.iloc[0]['radius_m'])
    print(f"\nSelected radius: {SELECTED_R} m")
else:
    SELECTED_R = 50
    print("\nNo radius met the 2% target. Using 50 m and documenting the limitation.")

with open(f'{EXP}/selected_radius.json', 'w') as f:
    json.dump({
        'selected_radius_m': SELECTED_R,
        'target_miss_rate_pct': 2,
        'selection_method': 'smallest radius with route-set coverage miss rate <= 2%',
        'sample_spacing_m': 10
    }, f, indent=2)

In [ ]:
all_obs = []
all_candidates = []

for split, base_dir in [('train', TRAIN), ('student_test', TEST)]:
    trip_dirs = sorted(d for d in Path(base_dir).iterdir() if d.is_dir())

    for trip_num, trip_dir in enumerate(trip_dirs, 1):
        trip_id = trip_dir.name

        track = pd.read_csv(
            next(trip_dir.glob('*.track')),
            sep=r'\s+', header=None,
            names=['lon', 'lat', 'timestamp']
        )
        nodes = pd.read_csv(
            next(trip_dir.glob('*.nodes')),
            sep=r'\s+', header=None,
            names=['lon', 'lat']
        )
        arcs = pd.read_csv(
            next(trip_dir.glob('*.arcs')),
            sep=r'\s+', header=None,
            names=['from_node', 'to_node']
        )
        route = pd.read_csv(
            next(trip_dir.glob('*.route')),
            sep=r'\s+', header=None
        ).iloc[:, 0].dropna().astype(int)

        proj = projections_by_trip[
            (projections_by_trip['split'] == split) &
            (projections_by_trip['trip_id'] == trip_id)
        ].iloc[0]

        transformer = Transformer.from_crs(
            'EPSG:4326', proj['crs'], always_xy=True
        )

        gps_x, gps_y = transformer.transform(
            track['lon'].to_numpy(), track['lat'].to_numpy()
        )
        node_x, node_y = transformer.transform(
            nodes['lon'].to_numpy(), nodes['lat'].to_numpy()
        )

        valid = (
            arcs['from_node'].between(0, len(nodes)-1) &
            arcs['to_node'].between(0, len(nodes)-1)
        )
        valid_arcs = arcs.loc[valid]
        arc_ids = valid_arcs.index.to_numpy()
        from_ids = valid_arcs['from_node'].to_numpy(dtype=int)
        to_ids = valid_arcs['to_node'].to_numpy(dtype=int)

        candidates = generate_candidates(
            gps_x, gps_y,
            node_x[from_ids], node_y[from_ids],
            node_x[to_ids], node_y[to_ids],
            arc_ids, radius=SELECTED_R, sample_spacing=10
        )

        route_ids = set(route.tolist())
        route_candidates = candidates[
            candidates['arc_id'].isin(route_ids)
        ]

        ref_map = (
            route_candidates.loc[
                route_candidates.groupby('obs_idx')['perp_distance_m'].idxmin()
            ].set_index('obs_idx')['arc_id'].to_dict()
            if len(route_candidates) else {}
        )

        obs = track.copy()
        obs['obs_idx'] = np.arange(len(obs))
        obs['trip_id'] = trip_id
        obs['split'] = split
        obs['x'] = gps_x
        obs['y'] = gps_y
        obs['ref_arc'] = obs['obs_idx'].map(ref_map).fillna(-1).astype(int)
        obs['is_ref'] = obs['ref_arc'] != -1
        all_obs.append(obs)

        if len(candidates):
            candidates['trip_id'] = trip_id
            candidates['split'] = split
            candidates['is_ref'] = candidates.apply(
                lambda r: int(ref_map.get(r['obs_idx'], -1)) == r['arc_id'],
                axis=1
            )
            all_candidates.append(candidates)

    print(f"{split}: processed {len(trip_dirs)} trips")

obs_df = pd.concat(all_obs, ignore_index=True)
candidates_df = pd.concat(all_candidates, ignore_index=True)

for split in ['train', 'student_test']:
    obs_df[obs_df['split'] == split].to_parquet(
        f'{DATA}/obs_{split}.parquet', index=False
    )
    candidates_df[candidates_df['split'] == split].to_parquet(
        f'{DATA}/candidates_{split}.parquet', index=False
    )

print("\nObservations:", len(obs_df))
print("Candidates:", len(candidates_df))
print("\nReference-label coverage:")
print(obs_df.groupby('split')['is_ref'].mean().mul(100).round(2))

In [ ]:
from sklearn.preprocessing import StandardScaler
import joblib

feature_parts = []

for split in ['train', 'student_test']:
    trip_ids = obs_df.loc[obs_df['split'] == split, 'trip_id'].unique()

    for trip_id in trip_ids:
        cf = candidates_df[
            (candidates_df['split'] == split) &
            (candidates_df['trip_id'] == trip_id)
        ].copy()

        if cf.empty:
            continue

        od = obs_df[
            (obs_df['split'] == split) &
            (obs_df['trip_id'] == trip_id)
        ].sort_values('obs_idx')

        trip_dir = Path(TRAIN if split == 'train' else TEST) / trip_id

        nodes = pd.read_csv(
            next(trip_dir.glob('*.nodes')),
            sep=r'\s+', header=None, names=['lon', 'lat']
        )
        arcs = pd.read_csv(
            next(trip_dir.glob('*.arcs')),
            sep=r'\s+', header=None,
            names=['from_node', 'to_node']
        )

        valid = (
            arcs['from_node'].between(0, len(nodes)-1) &
            arcs['to_node'].between(0, len(nodes)-1)
        )
        arcs = arcs.loc[valid]

        proj = projections_by_trip[
            (projections_by_trip['split'] == split) &
            (projections_by_trip['trip_id'] == trip_id)
        ].iloc[0]

        transformer = Transformer.from_crs(
            'EPSG:4326', proj['crs'], always_xy=True
        )
        nx, ny = transformer.transform(
            nodes['lon'].to_numpy(), nodes['lat'].to_numpy()
        )

        arc_ids = arcs.index.to_numpy()
        x1 = nx[arcs['from_node'].to_numpy(dtype=int)]
        y1 = ny[arcs['from_node'].to_numpy(dtype=int)]
        x2 = nx[arcs['to_node'].to_numpy(dtype=int)]
        y2 = ny[arcs['to_node'].to_numpy(dtype=int)]

        arc_length = np.hypot(x2-x1, y2-y1)
        arc_heading = np.arctan2(y2-y1, x2-x1)

        arc_features = pd.DataFrame({
            'arc_id': arc_ids,
            'arc_length_m': arc_length,
            'arc_heading': arc_heading
        })

        cf = cf.merge(arc_features, on='arc_id', how='left')

        gx = od['x'].to_numpy()
        gy = od['y'].to_numpy()

        dx = np.diff(gx, append=gx[-1])
        dy = np.diff(gy, append=gy[-1])

        if len(gx) > 1:
            dx[-1] = gx[-1] - gx[-2]
            dy[-1] = gy[-1] - gy[-2]

        gps_heading = np.arctan2(dy, dx)
        heading_map = dict(zip(od['obs_idx'], gps_heading))
        cf['gps_heading'] = cf['obs_idx'].map(heading_map)

        angle_diff = cf['gps_heading'] - cf['arc_heading']
        cf['heading_difference_deg'] = np.degrees(
            np.abs(np.arctan2(np.sin(angle_diff), np.cos(angle_diff)))
        )

        cf['candidate_rank'] = (
            cf.groupby('obs_idx')['perp_distance_m']
            .rank(method='first')
        )
        cf['candidate_count'] = (
            cf.groupby('obs_idx')['arc_id'].transform('count')
        )

        sorted_cf = cf.sort_values(
            ['obs_idx', 'perp_distance_m']
        )
        first = sorted_cf.groupby('obs_idx')['perp_distance_m'].first()
        second = sorted_cf.groupby('obs_idx')['perp_distance_m'].nth(1)
        gap = second.subtract(first, fill_value=0)

        cf['distance_gap_m'] = cf['obs_idx'].map(gap).fillna(0)
        cf = cf.drop(columns=['gps_heading', 'arc_heading'])

        feature_parts.append(cf)

    print(f"{split}: features generated for {len(trip_ids)} trips")

features_df = pd.concat(feature_parts, ignore_index=True)

FEATURES = [
    'perp_distance_m',
    'projection_fraction',
    'arc_length_m',
    'heading_difference_deg',
    'candidate_rank',
    'candidate_count',
    'distance_gap_m'
]

scaler = StandardScaler()
train_mask = features_df['split'] == 'train'

scaler.fit(features_df.loc[train_mask, FEATURES])
features_df[FEATURES] = scaler.transform(features_df[FEATURES])

joblib.dump(scaler, f'{ART}/scaler.joblib')

with open(f'{ART}/feature_list.json', 'w') as f:
    json.dump(FEATURES, f, indent=2)

with open(f'{ART}/config.json', 'w') as f:
    json.dump({
        'seed': 42,
        'candidate_radius_m': SELECTED_R,
        'sample_spacing_m': 10,
        'scaler': 'StandardScaler fitted on train only',
        'target': 'is_ref',
        'excluded_inputs': ['ref_arc', 'is_ref', 'route'],
        'features': FEATURES
    }, f, indent=2)

KEEP = [
    'trip_id', 'split', 'obs_idx', 'arc_id',
    'is_ref'
] + FEATURES

for split in ['train', 'student_test']:
    features_df.loc[
        features_df['split'] == split, KEEP
    ].to_parquet(f'{DATA}/features_{split}.parquet', index=False)

print("\nSaved model-ready datasets.")
print("Feature rows:", len(features_df))
print("Features:", FEATURES)
print("\nTrain:", (features_df['split'] == 'train').sum())
print("Student test:", (features_df['split'] == 'student_test').sum())
print("Scaler fitted on train rows only.")

In [ ]:
checks = []

for split in ['train', 'student_test']:
    df = features_df[features_df['split'] == split]

    checks.append({
        'split': split,
        'rows': len(df),
        'unique_trips': df['trip_id'].nunique(),
        'missing_features': int(df[FEATURES].isna().sum().sum()),
        'infinite_features': int(np.isinf(df[FEATURES].to_numpy()).sum()),
        'duplicate_obs_arc_pairs': int(
            df.duplicated(['trip_id', 'obs_idx', 'arc_id']).sum()
        ),
        'label_values': sorted(df['is_ref'].dropna().unique().tolist()),
        'ground_truth_columns_in_features': [
            c for c in ['ref_arc', 'route'] if c in df.columns
        ]
    })

checks_df = pd.DataFrame(checks)
display(checks_df)

# Confirm train/test separation and label integrity
assert set(features_df['split'].unique()) == {'train', 'student_test'}
assert not features_df[FEATURES].isna().any().any()
assert np.isfinite(features_df[FEATURES].to_numpy()).all()
assert not features_df.duplicated(
    ['trip_id', 'obs_idx', 'arc_id']
).any()
assert set(features_df['is_ref'].unique()).issubset({0, 1, False, True})
assert 'ref_arc' not in features_df.columns
assert 'route' not in features_df.columns

checks_df.to_csv(f'{EXP}/feature_validation.csv', index=False)

print("\nAll feature validation assertions passed.")

In [ ]:
contract = """# PathSnap Data Contract

## Dataset partitions
- train: 80 trips
- student_test: 10 trips
- Evaluator data is excluded from development.

## Model-ready files
- data/features_train.parquet
- data/features_student_test.parquet

Each row represents one GPS observation–candidate road arc pair.

## Identifiers and target
- trip_id: trip identifier; use for group-aware splitting.
- obs_idx: GPS observation index.
- arc_id: candidate road arc ID.
- is_ref: binary reference label; training target only.

## Model input features
1. perp_distance_m
2. projection_fraction
3. arc_length_m
4. heading_difference_deg
5. candidate_rank
6. candidate_count
7. distance_gap_m

Features are standardized using a StandardScaler fitted on training rows only.

## Leakage restrictions
Never use ref_arc, is_ref, route IDs, or other ground-truth information as model inputs.
Keep trip_id for grouping, not as a predictive feature.
Do not use student_test labels for fitting or tuning.

## Candidate generation
- Densified road-segment spatial index.
- Sample spacing: 10 metres.
- Selected candidate radius: 50 metres.
- The 2% route-set miss-rate target was not achieved at any tested radius.
- Reference labels are assigned to the nearest generated candidate belonging to the trip's route arc set.
- This is a spatial route-set proxy, not a time-aligned ground-truth match.

## Reproducibility
- Random seed: 42
- Projection: per-trip UTM CRS.
- Scaler: models/artifacts/scaler.joblib
- Feature list: models/artifacts/feature_list.json
- Configuration: models/artifacts/config.json
"""

with open(f'{OUT}/CONTRACT.md', 'w') as f:
    f.write(contract)

metadata = {
    'train_trips': 80,
    'student_test_trips': 10,
    'observations': int(len(obs_df)),
    'candidate_rows': int(len(candidates_df)),
    'feature_rows': int(len(features_df)),
    'feature_columns': FEATURES,
    'selected_radius_m': SELECTED_R,
    'train_label_coverage_pct': round(
        obs_df.loc[obs_df['split'] == 'train', 'is_ref'].mean() * 100, 2
    ),
    'student_test_label_coverage_pct': round(
        obs_df.loc[obs_df['split'] == 'student_test', 'is_ref'].mean() * 100, 2
    ),
    'seed': SEED
}

with open(f'{EXP}/pipeline_summary.json', 'w') as f:
    json.dump(metadata, f, indent=2)

print("Saved CONTRACT.md and pipeline_summary.json")

In [ ]:
import matplotlib.pyplot as plt

# 1. Candidate feature distributions (raw, before scaling)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for split, color in [('train', None), ('student_test', None)]:
    subset = candidates_df[candidates_df['split'] == split]
    axes[0].hist(
        subset['perp_distance_m'], bins=50, alpha=0.5,
        label=split
    )
    axes[1].hist(
        subset['projection_fraction'], bins=50, alpha=0.5,
        label=split
    )

axes[0].set_title('Perpendicular distance (train + student_test)')
axes[0].set_xlabel('Distance (m)')
axes[1].set_title('Projection fraction (train + student_test)')
axes[1].set_xlabel('Fraction along arc')
for ax in axes:
    ax.set_ylabel('Candidate count')
    ax.legend()
fig.tight_layout()
fig.savefig(f'{FIG}/candidate_feature_distributions.png', dpi=150)
plt.show()

# 2. GPS coordinate distribution by partition
fig, ax = plt.subplots(figsize=(9, 6))

for split in ['train', 'student_test']:
    subset = obs_df[obs_df['split'] == split]
    ax.scatter(
        subset['lon'], subset['lat'],
        s=1, alpha=0.35, label=f'{split} GPS points'
    )

ax.set_title('GPS coordinate distribution by partition')
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
ax.legend()
ax.grid(True, alpha=0.2)
fig.tight_layout()
fig.savefig(f'{FIG}/gps_partition_distribution.png', dpi=150)
plt.show()

# 3. Candidate count summary per observation
candidate_counts = (
    candidates_df.groupby(['split', 'trip_id', 'obs_idx'])
    .size()
    .rename('candidate_count')
    .reset_index()
)

candidate_summary = candidate_counts.groupby('split')['candidate_count'].agg(
    ['count', 'mean', 'median', 'min', 'max']
).reset_index()

candidate_summary.to_csv(
    f'{EXP}/candidate_count_summary.csv', index=False
)

display(candidate_summary)
print(f"\nEDA figures saved in {FIG}")

In [ ]:
import folium

trip_dir = sorted(
    d for d in Path(TRAIN).iterdir() if d.is_dir()
)[0]
trip_id = trip_dir.name

track = pd.read_csv(
    next(trip_dir.glob('*.track')),
    sep=r'\s+', header=None,
    names=['lon', 'lat', 'timestamp']
)
nodes = pd.read_csv(
    next(trip_dir.glob('*.nodes')),
    sep=r'\s+', header=None,
    names=['lon', 'lat']
)
arcs = pd.read_csv(
    next(trip_dir.glob('*.arcs')),
    sep=r'\s+', header=None,
    names=['from_node', 'to_node']
)
route = pd.read_csv(
    next(trip_dir.glob('*.route')),
    sep=r'\s+', header=None
).iloc[:, 0].dropna().astype(int)

center = [track['lat'].mean(), track['lon'].mean()]
m = folium.Map(location=center, zoom_start=13)

# GPS trajectory
gps_points = list(zip(track['lat'], track['lon']))
folium.PolyLine(
    gps_points, color='blue', weight=3,
    opacity=0.8, tooltip='GPS trajectory'
).add_to(m)

# Road network, limited to nearby nodes for readability
nearby = (
    nodes['lat'].between(track['lat'].min()-0.005, track['lat'].max()+0.005) &
    nodes['lon'].between(track['lon'].min()-0.005, track['lon'].max()+0.005)
)
nearby_ids = set(nodes.index[nearby])

network_group = folium.FeatureGroup(name='Road network')
for arc_id, arc in arcs.iterrows():
    a, b = int(arc['from_node']), int(arc['to_node'])
    if a in nearby_ids and b in nearby_ids:
        folium.PolyLine(
            [(nodes.loc[a, 'lat'], nodes.loc[a, 'lon']),
             (nodes.loc[b, 'lat'], nodes.loc[b, 'lon'])],
            color='gray', weight=1, opacity=0.35
        ).add_to(network_group)

network_group.add_to(m)

# Reference route
route_group = folium.FeatureGroup(name='Reference route')
for arc_id in route:
    if arc_id < 0 or arc_id >= len(arcs):
        continue
    arc = arcs.loc[arc_id]
    a, b = int(arc['from_node']), int(arc['to_node'])
    if a in nodes.index and b in nodes.index:
        folium.PolyLine(
            [(nodes.loc[a, 'lat'], nodes.loc[a, 'lon']),
             (nodes.loc[b, 'lat'], nodes.loc[b, 'lon'])],
            color='red', weight=3, opacity=0.8
        ).add_to(route_group)

route_group.add_to(m)
folium.LayerControl().add_to(m)

folium.Marker(center, tooltip='Training trip GPS centre').add_to(m)

m.save(f'{MAPS}/train_trip_{trip_id}_map.html')
print(f"Saved map for train trip {trip_id}")
print(f"Partition: train | GPS observations: {len(track)}")
m

In [ ]:
import os

required = [
    f'{OUT}/CONTRACT.md',
    f'{DATA}/obs_train.parquet',
    f'{DATA}/obs_student_test.parquet',
    f'{DATA}/candidates_train.parquet',
    f'{DATA}/candidates_student_test.parquet',
    f'{DATA}/features_train.parquet',
    f'{DATA}/features_student_test.parquet',
    f'{ART}/scaler.joblib',
    f'{ART}/feature_list.json',
    f'{ART}/config.json',
    f'{EXP}/inventory.csv',
    f'{EXP}/data_quality.csv',
    f'{EXP}/network_summary.csv',
    f'{EXP}/R_tuning_train.csv',
    f'{EXP}/selected_radius.json',
    f'{EXP}/feature_validation.csv',
    f'{EXP}/pipeline_summary.json',
    f'{FIG}/candidate_feature_distributions.png',
    f'{FIG}/gps_partition_distribution.png'
]

for path in required:
    print(('OK      ' if os.path.exists(path) else 'MISSING '), path)

print("\nMaps:", os.listdir(MAPS))

In [ ]:
import shutil
from google.colab import files

zip_path = shutil.make_archive(
    '/content/PathSnap_handoff',
    'zip',
    '/content',
    'PathSnap'
)

files.download(zip_path)